# 31 — Count-Min Sketch (Amazon FAR style)

Estimate item frequencies in sub-linear memory — like a Bloom filter but for counts. It **never
under-counts** (collisions only inflate), with a tunable error bound. Parts 1-3 build the core
(concurrency at Part 3); Parts 4-5 are stretch tasks (merge/union, then parallel build). Fill stubs,
run each test cell, peek at the collapsed `ref_` solutions only after trying.

`d` rows × `w` columns; `d` stable hashes (`_h`, provided). `add` bumps one cell per row; `estimate`
is the **min** across rows.

---

## Part 1 — Count-Min Sketch

`CountMinSketch(width, depth)` with `add(item, count=1)` (increment `table[r][h_r(item)]` in each row)
and `estimate(item)` (the **minimum** counter across rows).

**Lock down:** the min across rows is the estimate because every collision can only *add* to a cell —
so the estimate is `>=` the true count (an over-estimate), never below.

In [ ]:
import hashlib


def _h(item, r, w):
    return int(hashlib.md5(("%d:%s" % (r, item)).encode()).hexdigest(), 16) % w


class CountMinSketch:
    def __init__(self, width, depth):
        raise NotImplementedError

    def add(self, item, count=1):
        raise NotImplementedError

    def estimate(self, item):
        raise NotImplementedError

In [ ]:
# --- Part 1 tests ---
def _t1():
    import random
    cms = CountMinSketch(1000, 4)
    rng, truth = random.Random(0), {}
    for _ in range(2000):
        x = rng.choice(["a", "b", "c", "d", "e"])
        cms.add(x)
        truth[x] = truth.get(x, 0) + 1
    for x in truth:
        assert cms.estimate(x) >= truth[x]              # never under-counts
        assert cms.estimate(x) <= truth[x] + 50         # small over-count with a wide sketch
    print("Part 1 OK")

_t1()

## Part 2 — Sizing for an error bound

`optimal_size(epsilon, delta) -> (width, depth)`: `width = ceil(e / epsilon)`, `depth = ceil(ln(1 /
delta))`. This guarantees that over `N` updates, `estimate(x) <= true(x) + epsilon·N` with probability
`>= 1 - delta`.

**Lock down:** width controls the error magnitude (`epsilon`), depth controls the failure probability
(`delta`); verify the `+ epsilon·N` bound holds empirically.

In [ ]:
import math


def optimal_size(epsilon, delta):
    raise NotImplementedError

In [ ]:
# --- Part 2 tests ---
def _t2():
    import random
    eps, delta = 0.01, 0.01
    w, d = optimal_size(eps, delta)
    assert w >= math.ceil(math.e / eps) and d >= 1
    cms = CountMinSketch(w, d)
    rng, truth, N = random.Random(1), {}, 5000
    for _ in range(N):
        x = "k%d" % rng.randint(0, 99)
        cms.add(x)
        truth[x] = truth.get(x, 0) + 1
    for x in truth:
        assert cms.estimate(x) <= truth[x] + eps * N    # the CMS error guarantee
    print("Part 2 OK")

_t2()

## Part 3 — Thread-safe sketch

`ConcurrentCMS(width, depth)`: thread-safe `add`/`estimate`. Many threads increment the same item with
no lost updates.

**The invariant:** 8 threads each `add("x")` 1000 times gives `estimate("x") == 8000` (a lone item has
no collisions, so the estimate is exact). **Discuss:** each cell increment is a read-modify-write
needing the lock; per-row or atomic counters reduce contention; reads (`estimate`) are cheap.

In [ ]:
import threading


class ConcurrentCMS:
    def __init__(self, width, depth):
        raise NotImplementedError

    def add(self, item, count=1):
        raise NotImplementedError

    def estimate(self, item):
        raise NotImplementedError

In [ ]:
# --- Part 3 tests ---
def _t3():
    cms = ConcurrentCMS(2000, 4)

    def worker():
        for _ in range(1000):
            cms.add("x")

    ts = [threading.Thread(target=worker) for _ in range(8)]
    for t in ts: t.start()
    for t in ts: t.join()
    assert cms.estimate("x") == 8000                   # no lost increments (single item -> exact)
    print("Part 3 OK")

_t3()

## Part 4 (stretch) — Merge (union)

`merge(a, b) -> CountMinSketch`: combine two sketches **of the same dimensions** by adding their tables
cell-by-cell. The merged sketch estimates the combined counts.

**Discuss:** linearity — CMS supports merge because each cell is a sum, so partitioned streams can be
sketched separately and added (this is what makes Part 5 work); dimensions must match.

In [ ]:
def merge(a, b):
    raise NotImplementedError

In [ ]:
# --- Part 4 tests ---
def _t4():
    a, b = CountMinSketch(1000, 4), CountMinSketch(1000, 4)
    for _ in range(5):
        a.add("x")
    for _ in range(3):
        b.add("x")
    for _ in range(2):
        b.add("y")
    m = merge(a, b)
    assert m.estimate("x") >= 8 and m.estimate("y") >= 2
    assert m.estimate("x") <= 8 + 20                    # tight with a wide sketch
    print("Part 4 OK")

_t4()

## Part 5 (stretch) — Parallel build

`parallel_build(items, width, depth, num_procs) -> CountMinSketch`: build the sketch over a large item
list in parallel — each worker builds a table for its chunk (`cms_workers.build_table`), then sum the
tables (a merge). Estimates must match a sequentially-built sketch exactly.

**Discuss:** CMS tables add linearly, so partition → sketch → sum is exact (not approximate w.r.t. the
serial sketch); GIL → processes for the CPU-bound hashing; the stable hash keeps cells aligned.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import cms_workers


def parallel_build(items, width, depth, num_procs) -> "CountMinSketch":
    raise NotImplementedError

In [ ]:
# --- Part 5 tests ---
def _t5():
    items = ["k%d" % (i % 20) for i in range(1000)]
    cms = parallel_build(items, 500, 4, 4)
    seq = CountMinSketch(500, 4)
    for x in items:
        seq.add(x)
    for x in set(items):
        assert cms.estimate(x) == seq.estimate(x)       # identical tables -> identical estimates
    print("Part 5 OK")

_t5()

## Likely follow-ups
- Count-Min vs Count-Sketch (unbiased, supports decrements) vs Bloom (membership) vs HyperLogLog (cardinality).
- Heavy-hitters: CMS + a heap of candidates; conservative-update CMS to cut over-estimation.
- Choosing width/depth for a memory budget; point vs range queries (dyadic CMS).

---
## Reference solutions (don't peek until you've tried)

In [ ]:
import hashlib
import math
import threading
from concurrent.futures import ProcessPoolExecutor
import cms_workers


def _h(item, r, w):
    return int(hashlib.md5(("%d:%s" % (r, item)).encode()).hexdigest(), 16) % w


class RefCountMinSketch:
    def __init__(self, width, depth):
        self.w = width
        self.d = depth
        self.table = [[0] * width for _ in range(depth)]

    def add(self, item, count=1):
        for r in range(self.d):
            self.table[r][_h(item, r, self.w)] += count

    def estimate(self, item):
        return min(self.table[r][_h(item, r, self.w)] for r in range(self.d))


def ref_optimal_size(epsilon, delta):
    return math.ceil(math.e / epsilon), math.ceil(math.log(1.0 / delta))


class RefConcurrentCMS(RefCountMinSketch):
    def __init__(self, width, depth):
        super().__init__(width, depth)
        self.lock = threading.Lock()

    def add(self, item, count=1):
        with self.lock:
            super().add(item, count)

    def estimate(self, item):
        with self.lock:
            return super().estimate(item)


def ref_merge(a, b):
    m = RefCountMinSketch(a.w, a.d)
    for r in range(a.d):
        for c in range(a.w):
            m.table[r][c] = a.table[r][c] + b.table[r][c]
    return m


def ref_parallel_build(items, width, depth, num_procs):
    chunks = [items[i::num_procs] for i in range(num_procs)]
    cms = RefCountMinSketch(width, depth)
    with ProcessPoolExecutor(max_workers=num_procs) as ex:
        for table in ex.map(cms_workers.build_table, [(c, width, depth) for c in chunks]):
            for r in range(depth):
                row = table[r]
                for c in range(width):
                    cms.table[r][c] += row[c]
    return cms


_c = RefCountMinSketch(500, 4)
for _ in range(7):
    _c.add("a")
assert _c.estimate("a") >= 7
_w, _d = ref_optimal_size(0.01, 0.01); assert _w >= 271 and _d >= 1
_a = RefCountMinSketch(500, 4); _b = RefCountMinSketch(500, 4)
_a.add("z", 4); _b.add("z", 6); assert ref_merge(_a, _b).estimate("z") >= 10
_items = ["k%d" % (i % 5) for i in range(200)]
_p = ref_parallel_build(_items, 300, 4, 4); _s = RefCountMinSketch(300, 4)
for _it in _items:
    _s.add(_it)
assert all(_p.estimate(x) == _s.estimate(x) for x in set(_items))
print("reference solutions OK")